# Pràctica: el dataset Wine

**Optativa d'Aprenentatge automàtic — DAM/DAW 2n**

Fins ara heu vist els cinc models sobre el dataset d'Iris. És un bon punt de partida
perquè és petit i net, però també massa fàcil: amb només 4 columnes es pot mirar
gairebé tot en un gràfic.

Ara toca un dataset amb la mateixa forma —taula, classes, poques files— però una mica
més exigent: **Wine**, de scikit-learn. Són 178 vins de tres varietats diferents,
descrits per 13 mesures químiques (alcohol, magnesi, intensitat de color, prolina...).
Amb 13 columnes ja no es pot mirar tot d'un cop d'ull, i les escales són brutalment
diferents entre elles: `proline` va per centenars, `nonflavanoid_phenols` no arriba a
la unitat. És l'escenari perfecte per posar a prova el que vau aprendre sobre el
parany de les escales.

Els exercicis s'encadenen: les variables que crees en un exercici les fas servir en
el següent (`dades`, `X_entrena`, `X_examen`, `y_entrena`, `y_examen`...). Fes-los en
ordre.

## Les eines que necessites

Aquest quadern es fa amb aquestes eines. Cadascuna està explicada a
`EX_00_caixa_eines.ipynb`, amb la sintaxi i el parany que té. Aquí només tens la
llista: encaixar les peces és la feina de l'exercici.

De scikit-learn:
- `load_wine`
- `train_test_split`
- `StandardScaler`
- `KNeighborsClassifier`, `DecisionTreeClassifier`, `RandomForestClassifier`,
  `LogisticRegression`, `SVC`
- `DummyClassifier` i `confusion_matrix`, que necessites per a les comprovacions
  dels exercicis 5 i 6

De pandas i de numpy:
- `DataFrame` i `Series`
- `describe()`, `value_counts()`, `sort_values()`
- `.mean()` i `.std()` amb `axis=0`

De matplotlib, `plt.plot` per a les corbes de l'exercici 6 i
`Series.plot(kind="barh")` per a les barres de l'exercici 7.

## Els objectes que manegues, i què els pots demanar

Les eines de la llista de dalt no treballen soltes: cadascuna et torna un
**objecte**, i el pas següent de l'exercici surt de demanar-li alguna cosa a aquell
objecte. Aquí tens, de cada un, el que faràs servir de veritat en aquest quadern:
els **atributs** (sense parèntesis) i els **mètodes** (amb parèntesis).

**El `Bunch` que torna `load_wine()`** — `sklearn.utils.Bunch`, una mena de
diccionari al qual també pots accedir amb punt (`wine.data` és el mateix que
`wine["data"]`):
- `.data` — les 178×13 mesures químiques
- `.target` — les 178 etiquetes, en números (0, 1, 2)
- `.feature_names` — la llista amb els 13 noms de columna; és el que hi ha a `cols`
- `.target_names` — els 3 noms de varietat; indexar-la amb `.target` és el que
  converteix els números en noms
- `.frame` — només amb `as_frame=True`: el `DataFrame` amb les 13 columnes **i**
  `target`
- `.DESCR` — el text que descriu el dataset (`print(wine.DESCR)` per llegir-lo)

**L'`ndarray` de NumPy** — el que hi ha dins de `X`, `X_entrena`, `X_examen`,
`X_entrena_esc`:
- `.shape` — la tupla (files, columnes). És el primer que has de mirar després de
  partir les dades
- `.dtype` — el tipus dels valors que hi ha a dins
- `.T` — la matriu transposada
- `.mean(axis=0)` i `.std(axis=0)` — mitjana i desviació **per columna**; amb
  `axis=1` te les fa per fila, i sense `axis` te les fa de tots els valors alhora
- `.argmax()` — la **posició** del valor més gran, no el valor
- `.reshape()` — canvia la forma sense tocar els valors

**El `DataFrame` i la `Series` de pandas** — `dades`, `desviacions`, `importancies`,
`taula`:
- del `DataFrame`: `.shape`, `.columns`, `.dtypes`, `.index` (atributs) i
  `.head()`, `.describe()`, `.drop(columns=[...])`, `.sort_values(...)`,
  `.to_numpy()` (mètodes)
- de la `Series`: `.value_counts()`, `.sort_values()`, `.plot(kind="barh")`
- `dades[cols]` et torna un `DataFrame` amb aquestes columnes; `dades["classe"]`
  et torna una sola columna com a `Series`

**Un model de scikit-learn** — `KNeighborsClassifier`, `DecisionTreeClassifier`,
`RandomForestClassifier`, `LogisticRegression`, `SVC`. Els cinc tenen **la mateixa
interfície**, i això és mitja assignatura:
- `.fit(X, y)` — entrena amb aquestes dades; torna el model mateix
- `.predict(X)` — un array amb una etiqueta predita per fila
- `.score(X, y)` — la precisió sobre aquestes dades, en un sol número
- `.classes_` — les classes que ha vist en entrenar, ordenades
- `.n_features_in_` — quantes columnes esperava; si li dones un nombre diferent a
  `.predict()`, peta
- `.feature_importances_` — **només** els models d'arbre (`DecisionTreeClassifier`,
  `RandomForestClassifier`): un pes per columna, i tots junts sumen 1
- `.coef_` — **només** `LogisticRegression` (i un `SVC` lineal): un coeficient per
  columna i classe
- `.get_depth()` i `.get_n_leaves()` — només l'arbre

**L'`StandardScaler`** — no és un model, és un **transformador**, i per això té uns
altres mètodes:
- `.fit(X)`, `.transform(X)`, i els dos de cop amb `.fit_transform(X)`. No té
  `.predict()` ni `.score()`
- `.mean_` i `.scale_` — la mitjana i la desviació de cada columna que ha après de
  `X_entrena`. Són exactament les que fa servir després a `.transform(X_examen)`, i
  és per això que `X_examen_esc` no et surt amb mitjana 0

Dues regles que valen per a tot el quadern, perquè són les que fan petar el codi
sense que sembli que hi hagi res malament:

- **Un atribut va sense parèntesis i un mètode amb parèntesis.** `model.coef_` et
  dona els coeficients; `model.coef_()` dona `TypeError`, perquè no és una funció.
  I a l'inrevés: el `.describe` d'un DataFrame sense parèntesis no calcula res,
  t'ensenya el mètode.
- **El guió baix al final vol dir «això ho he après de les dades».** `.coef_`,
  `.feature_importances_` i `.classes_` **no existeixen abans de cridar `.fit()`**:
  si els demanes abans, salta un error. El mateix passa amb els atributs apresos
  de qualsevol transformador.

Per interrogar qualsevol d'aquests objectes pel teu compte —`type()`, `dir()`,
`objecte.metode?` i l'autocompletat amb TAB— tens
`FO_00_objectes_i_autocompletar.ipynb`.

### Exercici 1 — Explora Wine

Carrega el dataset amb `load_wine(as_frame=True)` i munta un DataFrame `dades` amb
les 13 columnes químiques més una columna `classe` amb el nom de la varietat
(`wine.target_names[wine.target]`; descarta la columna `target` del `.frame`, que fa
la mateixa feina en números).

Digues quantes files i columnes té, quantes classes hi ha i si estan equilibrades.
Fes un `describe()` i fixa't en el rang (mínim i màxim) de `proline` i de
`nonflavanoid_phenols`.

*Com saps que ho has fet bé: 178 files, 14 columnes (13 + `classe`), 3 classes
(59 / 71 / 48 mostres). `proline` va de 278 a 1680; `nonflavanoid_phenols` va de
0,13 a 0,66 — mateix dataset, escales completament diferents.*

In [ ]:
# Exercici 1
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)

# 1. Crea "dades": el frame de wine (wine.frame) sense la columna "target"
#    (.drop(columns=...)), i amb una columna "classe" nova
#    (wine.target_names[wine.target])
dades = None

# 2. Quantes files i columnes té "dades"? (dades.shape)

# 3. Quantes classes hi ha i quantes mostres de cada
#    (dades["classe"].value_counts())?

# 4. Fes un dades.describe() de les columnes numèriques i mira el min i el
#    max de "proline" i "nonflavanoid_phenols"

### Exercici 2 — El flux complet

Separa `dades` en `X` (les 13 columnes químiques) i `y` (la columna `classe`). Fes
el split 70/30 amb `train_test_split(..., test_size=0.3, random_state=42,
stratify=y)`, entrena un `DecisionTreeClassifier(random_state=42)` i mesura la
precisió a l'examen.

*Com saps que ho has fet bé: comprova la partició abans de creure't cap número.
`X_entrena` ha de tenir 124 files i `X_examen` 54, amb 13 columnes totes dues, i
124 + 54 ha de fer 178. Comprova també que `stratify` ha fet la seva feina: les
proporcions de classe de `y_examen` (`y_examen.value_counts(normalize=True)`) han de
quedar molt a prop de les de `y` sencer. Si la partició està torta, cap precisió que
mesuris després no es pot comparar amb res. La precisió que t'hagi sortit a l'examen,
apunta-la: la faràs servir com a referència a la resta del quadern.*

In [ ]:
# Exercici 2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

cols = wine.feature_names

# 1. X = les 13 columnes de "dades" en un array (dades[cols].to_numpy());
#    y = la columna "classe"
X = None
y = None

# 2. Split 70/30 amb train_test_split(X, y, test_size=0.3,
#    random_state=42, stratify=y)
X_entrena, X_examen, y_entrena, y_examen = None, None, None, None

# 3. Entrena un DecisionTreeClassifier(random_state=42) amb
#    .fit(X_entrena, y_entrena) i mesura la precisió a l'examen amb
#    .score(X_examen, y_examen)

### Exercici 3 — k-NN sense escalar i escalat

Amb el mateix `X_entrena`/`X_examen` de l'exercici 2 (sense tocar-los), entrena un
`KNeighborsClassifier(n_neighbors=5)` **sense escalar** i anota la precisió a
l'examen. Després aplica `StandardScaler` (ajustat només amb `X_entrena`) i entrena
un altre `KNeighborsClassifier(n_neighbors=5)` amb les dades escalades.

Compara les dues precisions i escriu en un comentari, amb les teves paraules, què
explica la diferència que t'hagi sortit — relaciona-ho amb el parany de les
escales que vau veure amb Iris al quadern de k-NN.

*Com saps que ho has fet bé: comprova que l'escalat s'ha aplicat de veritat.
`X_entrena_esc.mean(axis=0)` ha de donar pràcticament zero a cada columna, i
`X_entrena_esc.std(axis=0)` pràcticament u. A `X_examen_esc` no hi sortiran
exactament 0 i 1, i és correcte que no hi surtin: l'escalador s'ha ajustat només amb
l'entrenament. Si l'entrenament escalat no et dona mitjana 0 i desviació 1, l'escalat
no s'ha aplicat com et penses i la comparació de precisions no val res. Un cop
comprovat, compara les dues precisions i respon tu quina guanya i per quant.*

In [ ]:
# Exercici 3
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. KNeighborsClassifier(n_neighbors=5) sobre X_entrena/X_examen SENSE
#    escalar: .fit() i després .score()
precisio_knn_sense = None

# 2. StandardScaler ajustat amb X_entrena (.fit_transform) i aplicat a
#    X_examen (.transform, que NO es torna a ajustar). Mira'n després
#    .mean_ i .scale_
escalador = StandardScaler()
X_entrena_esc = None
X_examen_esc = None

# 3. KNeighborsClassifier(n_neighbors=5) sobre les dades escalades
#    (.fit i .score, igual que al punt 1)
precisio_knn_esc = None

print(f"Sense escalar: {precisio_knn_sense}")
print(f"Escalat:       {precisio_knn_esc}")

# Que explica la diferencia entre les dues precisions? (escriu-ho aqui)

### Exercici 4 — Quina columna té la culpa

Calcula la desviació estàndard de cada columna de `X_entrena` (sense escalar) i
ordena-les de més gran a més petita. Aquestes són, en unitats originals, les
columnes que dominaran (o no) el càlcul de distàncies de k-NN abans d'escalar.

*Com saps que ho has fet bé: `proline` té una desviació al voltant de 307, per davant
de totes les altres — la següent (`magnesium`) no arriba a 16. `nonflavanoid_phenols`
tanca la llista amb una desviació d'uns 0,12.*

In [ ]:
# Exercici 4
import pandas as pd

# Calcula la desviació estàndard de cada columna de X_entrena
# (X_entrena.std(axis=0)), posa-la en una pd.Series indexada per "cols"
# i ordena-la de més gran a més petita (.sort_values(ascending=False))
desviacions = None

### Exercici 5 — Compara els cinc models

Sobre Wine **escalat** (`X_entrena_esc`/`X_examen_esc` de l'exercici 3), entrena els
cinc models que coneixeu: `KNeighborsClassifier(n_neighbors=5)`,
`DecisionTreeClassifier(random_state=42)`,
`RandomForestClassifier(n_estimators=100, random_state=42)`,
`LogisticRegression(random_state=42, max_iter=10000)` i `SVC(random_state=42)`.
Munta una taula (`DataFrame`) amb el model i la seva precisió a l'examen, ordenada
de major a menor.

Compara aquest rànquing amb el que recordeu d'Iris. És el mateix?

*Com saps que ho has fet bé: dues comprovacions, i cap de les dues no és el rànquing.
La taula ha de tenir cinc files, i cap precisió no pot quedar per sota del que fa un
`DummyClassifier(strategy="most_frequent")` sobre aquestes mateixes dades (ronda el
39 %, que és el pes de la classe més nombrosa): un model que no guanya el model tonto
està mal entrenat. I una coherència amb l'exercici 2: l'arbre de decisió d'aquesta
taula ha de donar exactament la mateixa precisió que allà, encara que aquí les dades
vagin escalades. Un arbre talla per llindars, i l'escalat no canvia l'ordre dels
valors dins d'una columna, així que no l'afecta gens; si et surt diferent, has
barrejat les X en algun lloc. El rànquing, i la comparació amb el d'Iris, els respons
tu.*

In [ ]:
# Exercici 5
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "k-NN": KNeighborsClassifier(n_neighbors=5),
    "Arbre de decisió": DecisionTreeClassifier(random_state=42),
    "Bosc aleatori": RandomForestClassifier(n_estimators=100, random_state=42),
    "Regressió logística": LogisticRegression(random_state=42, max_iter=10000),
    "SVM": SVC(random_state=42),
}

# Recorre models.items(): entrena cada model sobre X_entrena_esc/y_entrena
# amb .fit(), mesura la precisió a X_examen_esc/y_examen amb .score() i ves
# afegint la parella (nom, precisió) a "resultats". Ordena la taula amb
# .sort_values("precisio", ascending=False)
resultats = []

taula = pd.DataFrame(resultats, columns=["model", "precisio"])
taula

# Compara amb el rànquing d'Iris: és el mateix? (escriu-ho aquí)

### Exercici 6 — Profunditat de l'arbre

Sobre les dades **sense escalar** (els arbres no en necessiten), entrena un
`DecisionTreeClassifier(max_depth=d, random_state=42)` per a cada `d` d'1 a 15.
Guarda la precisió a l'entrenament i a l'examen de cada arbre i dibuixa les dues
corbes juntes. A partir de quina profunditat comença el sobreajust (la precisió
d'entrenament segueix pujant però la d'examen ja no)?

*Com saps que ho has fet bé: dues comprovacions d'estructura, cap número de precisió.
Primera: la corba d'entrenament no pot baixar mai quan puja la profunditat —més
profunditat és més llibertat per encertar el que ja ha vist—, i a partir d'una certa
`d` les dues corbes es queden planes, perquè l'arbre ja ha crescut tot el que podia i
afegir-hi `max_depth` no canvia res. Si te'n baixa cap tram, tens un error al bucle.
Segona: amb `max_depth=1` l'arbre té dues fulles per a tres classes, així que hi ha
una classe que no pot predir ni una vegada; comprova-ho amb `confusion_matrix` (de
`sklearn.metrics`) i hi trobaràs una columna sencera a zero. On comença el
sobreajust, i quant de marge et deixa, ho llegeixes tu al gràfic.*

In [ ]:
# Exercici 6
import matplotlib.pyplot as plt

profunditats = range(1, 16)
precisions_entrena = []
precisions_examen = []

# 1. Per cada "d" a profunditats: entrena un DecisionTreeClassifier(max_depth=d,
#    random_state=42) amb .fit(), i afegeix (.append()) .score(X_entrena,
#    y_entrena) i .score(X_examen, y_examen) a les dues llistes

# 2. Dibuixa les dues corbes juntes: un plt.plot de profunditats contra cada
#    llista (cada un amb el seu label=...), plt.xlabel, plt.ylabel i
#    plt.legend() per distingir entrenament d'examen

### Exercici 7 — Importància de les variables

Entrena un `RandomForestClassifier(n_estimators=100, random_state=42)` amb les dades
sense escalar (els boscos tampoc en necessiten) i mira `feature_importances_`. Fes
un gràfic de barres horitzontal amb les 13 columnes ordenades de més a menys
importància. Quines tres manen?

*Com saps que ho has fet bé: `alcohol`, `color_intensity` i `flavanoids` són les tres
primeres, totes al voltant del 16 % d'importància cadascuna — i no coincideixen amb
les columnes que dominaven la distància de k-NN a l'exercici 4 (`proline`,
`magnesium`). Un bosc aleatori no treballa amb distàncies: l'escala no li afecta.*

In [ ]:
# Exercici 7
bosc_final = RandomForestClassifier(n_estimators=100, random_state=42)

# 1. Entrena bosc_final amb X_entrena/y_entrena (sense escalar), amb .fit()

# 2. Munta una pd.Series amb bosc_final.feature_importances_ indexada per
#    "cols" i ordena-la de més a menys importància (.sort_values())
importancies = None

# 3. Gràfic de barres horitzontal (importancies.plot(kind="barh"))

## Resum

Has fet el mateix recorregut que amb Iris, però amb un dataset una mica més exigent:
13 columnes en lloc de 4, i escales que van d'unitats a centenars. El parany de les
escales de k-NN, que amb Iris només es notava si el provocàveu artificialment
(multiplicant una columna x100), aquí surt sol només carregant les dades tal com
venen.

Queda't amb dues idees per als propers quaderns:
- **Mira sempre `describe()` abans d'entrenar res.** Els rangs et diuen si cal
  escalar abans de començar.
- **"Quina columna importa" depèn del model.** Per a k-NN importa qui domina la
  distància (`proline`); per a un bosc aleatori importa qui separa millor les
  classes (`alcohol`, `color_intensity`, `flavanoids`). No són la mateixa pregunta.